# No.8 CT フィルタード逆投影 — 全8パターン

投影本数(128/256) × フィルタ(あり/なし) × 逆投影(あり/なし) = 8通り

| # | 投影本数 | フィルタ | 逆投影 | 出力 |
|---|---|---|---|---|
| 1 | 128 | なし | なし | サイノグラム画像 |
| 2 | 256 | なし | なし | サイノグラム画像 |
| 3 | 128 | あり | なし | フィルタ後サイノグラム |
| 4 | 256 | あり | なし | フィルタ後サイノグラム |
| 5 | 128 | なし | あり | フィルタなし逆投影 |
| 6 | 256 | なし | あり | フィルタなし逆投影 |
| 7 | 128 | あり | あり | FBP再構成 |
| 8 | 256 | あり | あり | FBP再構成 |

In [ ]:
import numpy as np
import skimage.io
import skimage.transform
import skimage.data
import matplotlib.pyplot as plt
import japanize_matplotlib
import os

os.makedirs("output", exist_ok=True)

## ファントム画像の準備と順方向投影

In [ ]:
# Crnl256.dat の読み込み
def load_dat(filepath, n_angles=256, n_samples=256):
    sinogram = np.zeros((n_samples, n_angles))
    with open(filepath) as f:
        lines = [l.strip() for l in f if l.strip()]
    idx = 0
    for angle_idx in range(n_angles):
        for sample_idx in range(n_samples):
            if idx < len(lines):
                parts = lines[idx].split()
                if len(parts) >= 2:
                    sinogram[sample_idx, angle_idx] = float(parts[1])
                idx += 1
    return sinogram

# Shepp-Logan ファントムから投影を生成（128本と256本）
phantom = skimage.data.shepp_logan_phantom()
phantom = skimage.transform.resize(phantom, (256, 256))

n_angles_list = [128, 256]
sinograms = {}
for n_ang in n_angles_list:
    angles = np.linspace(0, 180, n_ang, endpoint=False)
    sino = skimage.transform.radon(phantom, theta=angles)
    sinograms[n_ang] = (sino, angles)
    print(f"投影本数 {n_ang}: sinogram shape = {sino.shape}")

## ヘルパー関数

In [ ]:
def ramp_filter(sinogram):
    """周波数領域でランプフィルタを適用（Shepp-Logan型）"""
    n_samples = sinogram.shape[0]
    # 周波数軸
    freq = np.fft.fftfreq(n_samples)
    # Shepp-Logan フィルタ: |f| * sinc(f)
    filt = np.abs(freq)
    # ゼロ周波数を小さい値に
    filt[0] = filt[1] / 4

    filtered = np.zeros_like(sinogram)
    for i in range(sinogram.shape[1]):
        proj_fft = np.fft.fft(sinogram[:, i])
        filtered[:, i] = np.real(np.fft.ifft(proj_fft * filt))
    return filtered

def simple_backprojection(sinogram, angles, size=256):
    """フィルタなし逆投影（単純な逆投影）"""
    recon = np.zeros((size, size))
    center = size / 2.0
    for i, angle in enumerate(angles):
        theta = np.deg2rad(angle)
        cos_t = np.cos(theta)
        sin_t = np.sin(theta)
        proj = sinogram[:, i]
        proj_center = len(proj) / 2.0
        for x in range(size):
            for y in range(size):
                t = (x - center) * cos_t + (y - center) * sin_t + proj_center
                t_idx = int(round(t))
                if 0 <= t_idx < len(proj):
                    recon[x, y] += proj[t_idx]
    return recon

def to_pgm_ct(img, filename):
    """画像を正規化してpgm保存"""
    normalized = img - img.min()
    if normalized.max() > 0:
        normalized = normalized / normalized.max() * 255
    img_u8 = normalized.astype(np.uint8)
    skimage.io.imsave(filename, img_u8)
    return img_u8

## 課題1: 投影（サイノグラム画像化）— パターン1,2

投影本数128と256のサイノグラムを画像化。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for idx, n_ang in enumerate(n_angles_list):
    sino, angles = sinograms[n_ang]
    img = to_pgm_ct(sino, f"output/proj{n_ang}.pgm")
    axes[idx].imshow(img, cmap="gray", aspect="auto")
    axes[idx].set_title(f"投影本数 {n_ang}")
    axes[idx].set_xlabel("Angle")
    axes[idx].set_ylabel("Detector")
fig.suptitle("課題1: サイノグラム（フィルタなし・逆投影なし）", fontsize=14)
fig.tight_layout()
fig.savefig("output/pattern1_2_sinogram.png", dpi=150)
plt.show()

## 課題2: フィルタ処理（逆投影なし）— パターン3,4

In [ ]:
filtered_sinos = {}
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for idx, n_ang in enumerate(n_angles_list):
    sino, angles = sinograms[n_ang]
    filtered = ramp_filter(sino)
    filtered_sinos[n_ang] = filtered
    img = to_pgm_ct(filtered, f"output/proj{n_ang}_filtered.pgm")
    axes[idx].imshow(img, cmap="gray", aspect="auto")
    axes[idx].set_title(f"投影本数 {n_ang}")
    axes[idx].set_xlabel("Angle")
    axes[idx].set_ylabel("Detector")
fig.suptitle("課題2: フィルタ後サイノグラム（逆投影なし）", fontsize=14)
fig.tight_layout()
fig.savefig("output/pattern3_4_filtered.png", dpi=150)
plt.show()

## 課題3: 逆投影（フィルタなし）— パターン5,6

skimage.transform.iradon の filter_name=None で逆投影のみ実行。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for idx, n_ang in enumerate(n_angles_list):
    sino, angles = sinograms[n_ang]
    recon = skimage.transform.iradon(sino, theta=angles, filter_name=None)
    img = to_pgm_ct(recon, f"output/bp{n_ang}_nofilter.pgm")
    axes[idx].imshow(img, cmap="gray")
    axes[idx].set_title(f"投影本数 {n_ang}")
    axes[idx].axis("off")
fig.suptitle("課題3: フィルタなし逆投影", fontsize=14)
fig.tight_layout()
fig.savefig("output/pattern5_6_bp_nofilter.png", dpi=150)
plt.show()

## 課題3: FBP（フィルタあり逆投影）— パターン7,8

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for idx, n_ang in enumerate(n_angles_list):
    sino, angles = sinograms[n_ang]
    recon = skimage.transform.iradon(sino, theta=angles, filter_name="shepp-logan")
    img = to_pgm_ct(recon, f"output/fbp{n_ang}.pgm")
    axes[idx].imshow(img, cmap="gray")
    axes[idx].set_title(f"投影本数 {n_ang}")
    axes[idx].axis("off")
fig.suptitle("課題3: FBP再構成（フィルタあり逆投影）", fontsize=14)
fig.tight_layout()
fig.savefig("output/pattern7_8_fbp.png", dpi=150)
plt.show()

## 全8パターン一覧

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
titles = [
    ("投影128\nフィルタなし・逆投影なし", f"output/proj128.pgm"),
    ("投影256\nフィルタなし・逆投影なし", f"output/proj256.pgm"),
    ("投影128\nフィルタあり・逆投影なし", f"output/proj128_filtered.pgm"),
    ("投影256\nフィルタあり・逆投影なし", f"output/proj256_filtered.pgm"),
    ("投影128\nフィルタなし・逆投影あり", f"output/bp128_nofilter.pgm"),
    ("投影256\nフィルタなし・逆投影あり", f"output/bp256_nofilter.pgm"),
    ("投影128\nFBP（フィルタ+逆投影）", f"output/fbp128.pgm"),
    ("投影256\nFBP（フィルタ+逆投影）", f"output/fbp256.pgm"),
]
for idx, (title, path) in enumerate(titles):
    row, col = idx // 4, idx % 4
    img = skimage.io.imread(path)
    axes[row, col].imshow(img, cmap="gray")
    axes[row, col].set_title(title, fontsize=10)
    axes[row, col].axis("off")

fig.suptitle("全8パターン一覧", fontsize=16)
fig.tight_layout()
fig.savefig("output/all_8_patterns.png", dpi=150)
plt.show()
print("全8パターンのpgm画像を output/ に保存しました")